# Data Preprocessing


Import libraries and paths

In [1]:
import os
import numpy as np
import pandas as pd

from IPython.display import display

# Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Project paths
BASE_DIR = "/content/drive/MyDrive/ML Assignment"
SPLIT_DIR = os.path.join(BASE_DIR, "splitted_dataset")

print("Project directory:", BASE_DIR)
print("Split directory:", SPLIT_DIR)
print("Files:", os.listdir(SPLIT_DIR))

Mounted at /content/drive
Project directory: /content/drive/MyDrive/ML Assignment
Split directory: /content/drive/MyDrive/ML Assignment/splitted_dataset
Files: ['train.csv', 'validation.csv', 'test.csv']


Load Splitted datasets

In [2]:
train_df = pd.read_csv(
    os.path.join(SPLIT_DIR, "train.csv")
)

val_df = pd.read_csv(
    os.path.join(SPLIT_DIR, "validation.csv")
)

test_df = pd.read_csv(
    os.path.join(SPLIT_DIR, "test.csv")
)

print("Training shape   :", train_df.shape)
print("Validation shape :", val_df.shape)
print("Test shape       :", test_df.shape)

Training shape   : (18183, 121)
Validation shape : (3873, 121)
Test shape       : (3923, 121)


In [3]:
# Preserve match IDs separately for safe alignment in feature engineering

train_ids = train_df["id"].copy()
val_ids = val_df["id"].copy()
test_ids = test_df["id"].copy()

# Verify IDs are valid
assert train_ids.is_unique, "Duplicate match IDs found in training data."
assert val_ids.is_unique, "Duplicate match IDs found in validation data."
assert test_ids.is_unique, "Duplicate match IDs found in test data."

assert len(train_ids) == len(train_df)
assert len(val_ids) == len(val_df)
assert len(test_ids) == len(test_df)

print("Match IDs preserved successfully.")
print("Training IDs   :", len(train_ids))
print("Validation IDs :", len(val_ids))
print("Test IDs       :", len(test_ids))

Match IDs preserved successfully.
Training IDs   : 18183
Validation IDs : 3873
Test IDs       : 3923


Verify the target

In [4]:
TARGET = "target_outcome"

print("Target column:", TARGET)

for name, df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name} target distribution:")
    display(
        df[TARGET]
        .value_counts()
        .rename_axis("Outcome")
        .reset_index(name="Count")
    )

Target column: target_outcome

Training target distribution:


,Outcome,Count
0,Home Win,8433
1,Away Win,5126
2,Draw,4624



Validation target distribution:


,Outcome,Count
0,Home Win,1753
1,Away Win,1147
2,Draw,973



Test target distribution:


,Outcome,Count
0,Home Win,1731
1,Away Win,1193
2,Draw,999


Verify chronological split

In [5]:
for name, df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    print(f"{name}:")
    print("  Start:", df["date"].min())
    print("  End  :", df["date"].max())

Training:
  Start: 2008-07-18 00:00:00
  End  : 2014-02-10 00:00:00
Validation:
  Start: 2014-02-11 00:00:00
  End  : 2015-04-10 00:00:00
Test:
  Start: 2015-04-11 00:00:00
  End  : 2016-05-25 00:00:00


In [6]:
print(
    "Training end < Validation start:",
    train_df["date"].max() <= val_df["date"].min()
)

print(
    "Validation end < Test start:",
    val_df["date"].max() <= test_df["date"].min()
)

Training end < Validation start: True
Validation end < Test start: True


In [7]:
for boundary_date in [
    val_df["date"].min(),
    test_df["date"].min()
]:
    print("=" * 60)
    print("Boundary date:", boundary_date)

    print(
        "Training:",
        (train_df["date"] == boundary_date).sum()
    )

    print(
        "Validation:",
        (val_df["date"] == boundary_date).sum()
    )

    print(
        "Test:",
        (test_df["date"] == boundary_date).sum()
    )

Boundary date: 2014-02-11 00:00:00
Training: 0
Validation: 6
Test: 0
Boundary date: 2015-04-11 00:00:00
Training: 0
Validation: 0
Test: 38


Create explicit leakage list

In [8]:
LEAKAGE_COLUMNS = [
    "home_team_goal",
    "away_team_goal",
    "target_outcome",
    "total_goals",
    "goal_difference"
]

print("Leakage columns:")
for col in LEAKAGE_COLUMNS:
    print("-", col)

Leakage columns:
- home_team_goal
- away_team_goal
- target_outcome
- total_goals
- goal_difference


Identify non-predictive columns

In [9]:
id_columns = [
    col for col in train_df.columns
    if col.lower() == "id"
    or col.lower().endswith("_id")
    or col.lower().endswith("_api_id")
]

print("Potential identifier columns:")
for col in id_columns:
    print("-", col)

Potential identifier columns:
- id
- country_id
- league_id
- match_api_id
- home_team_api_id
- away_team_api_id
- match_id


In [10]:
id_summary = []

for col in id_columns:
    id_summary.append({
        "Column": col,
        "Unique Values": train_df[col].nunique(),
        "Training Rows": len(train_df),
        "Unique Ratio": round(train_df[col].nunique() / len(train_df), 4)
    })

display(pd.DataFrame(id_summary))

,Column,Unique Values,Training Rows,Unique Ratio
0,id,18183,18183,1.0000
1,country_id,11,18183,0.0006
2,league_id,11,18183,0.0006
3,match_api_id,18183,18183,1.0000
4,home_team_api_id,274,18183,0.0151
5,away_team_api_id,274,18183,0.0151
6,match_id,18183,18183,1.0000


In [11]:
IDENTIFIER_COLUMNS = [
    "id",
    "match_id",
    "country_id",
    "league_id",
    "match_api_id",
    "home_team_api_id",
    "away_team_api_id"
]

IDENTIFIER_COLUMNS = [
    col for col in IDENTIFIER_COLUMNS
    if col in train_df.columns
]

print("Identifiers selected for removal:")
for col in IDENTIFIER_COLUMNS:
    print("-", col)

Identifiers selected for removal:
- id
- match_id
- country_id
- league_id
- match_api_id
- home_team_api_id
- away_team_api_id


Create preprocessing exclusion list

In [12]:
EXCLUDE_COLUMNS = sorted(
    set(LEAKAGE_COLUMNS + IDENTIFIER_COLUMNS)
    .intersection(train_df.columns)
)

print("Total columns excluded:", len(EXCLUDE_COLUMNS))

for col in EXCLUDE_COLUMNS:
    print("-", col)

Total columns excluded: 10
- away_team_api_id
- away_team_goal
- country_id
- home_team_api_id
- home_team_goal
- id
- league_id
- match_api_id
- match_id
- target_outcome


Create working datasets

In [13]:
X_train_raw = train_df.drop(columns=EXCLUDE_COLUMNS).copy()
X_val_raw = val_df.drop(columns=EXCLUDE_COLUMNS).copy()
X_test_raw = test_df.drop(columns=EXCLUDE_COLUMNS).copy()

print("Training:", X_train_raw.shape)
print("Validation:", X_val_raw.shape)
print("Test:", X_test_raw.shape)

Training: (18183, 111)
Validation: (3873, 111)
Test: (3923, 111)


In [14]:
# Separately store y
y_train = train_df[TARGET].copy()
y_val = val_df[TARGET].copy()
y_test = test_df[TARGET].copy()

print("X_train:", X_train_raw.shape)
print("y_train:", y_train.shape)

X_train: (18183, 111)
y_train: (18183,)


Check remaining columns

In [15]:
column_summary = pd.DataFrame({
    "Column": X_train_raw.columns,
    "Data Type": X_train_raw.dtypes.astype(str),
    "Missing Values": X_train_raw.isnull().sum().values,
    "Missing %": (
        X_train_raw.isnull().mean() * 100
    ).round(2).values,
    "Unique Values": X_train_raw.nunique().values
})

display(column_summary)

,Column,Data Type,Missing Values,Missing %,Unique Values
season,season,object,0,0.00,6
stage,stage,int64,0,0.00,38
date,date,datetime64[ns],0,0.00,1198
home_player_X1,home_player_X1,float64,1821,10.01,3
home_player_X2,home_player_X2,float64,1821,10.01,8
...,...,...,...,...,...
BSA,BSA,float64,4022,22.12,96
league_name,league_name,object,0,0.00,11
country_name,country_name,object,0,0.00,11
home_team,home_team,object,0,0.00,272


Missing-value assessment

In [16]:
def missing_report(df, name):
    result = pd.DataFrame({
        "Column": df.columns,
        "Missing Values": df.isnull().sum().values,
        "Missing %": (df.isnull().mean() * 100).round(2).values
    })

    result = result[result["Missing Values"] > 0]
    result = result.sort_values("Missing %", ascending=False)

    print(f"\n{name} missing-value report")
    display(result)

missing_report(X_train_raw, "Training")
missing_report(X_val_raw, "Validation")
missing_report(X_test_raw, "Test")


Training missing-value report


,Column,Missing Values,Missing %
91,PSA,13779,75.78
90,PSD,13779,75.78
89,PSH,13779,75.78
69,goal,8929,49.11
73,card,8929,49.11
...,...,...,...
55,home_player_9,1247,6.86
58,away_player_1,1219,6.70
47,home_player_1,1210,6.65
64,away_player_7,1205,6.63



Validation missing-value report


,Column,Missing Values,Missing %
101,GBH,3873,100.00
102,GBD,3873,100.00
106,BSA,3873,100.00
105,BSD,3873,100.00
103,GBA,3873,100.00
104,BSH,3873,100.00
96,SJD,2562,66.15
97,SJA,2562,66.15
95,SJH,2562,66.15
73,card,1557,40.20



Test missing-value report


,Column,Missing Values,Missing %
103,GBA,3923,100.00
104,BSH,3923,100.00
101,GBH,3923,100.00
96,SJD,3923,100.00
106,BSA,3923,100.00
97,SJA,3923,100.00
95,SJH,3923,100.00
105,BSD,3923,100.00
102,GBD,3923,100.00
73,card,1276,32.53


Compare missing-value patterns

In [17]:
missing_comparison = pd.DataFrame({
    "Training Missing": X_train_raw.isnull().sum(),
    "Validation Missing": X_val_raw.isnull().sum(),
    "Test Missing": X_test_raw.isnull().sum()
})

missing_comparison["Total Missing"] = (
    missing_comparison["Training Missing"]
    + missing_comparison["Validation Missing"]
    + missing_comparison["Test Missing"]
)

display(
    missing_comparison.sort_values(
        "Total Missing",
        ascending=False
    )
)

,Training Missing,Validation Missing,Test Missing,Total Missing
PSH,13779,521,511,14811
PSD,13779,521,511,14811
PSA,13779,521,511,14811
BSD,4022,3873,3923,11818
BSH,4022,3873,3923,11818
...,...,...,...,...
stage,0,0,0,0
league_name,0,0,0,0
country_name,0,0,0,0
home_team,0,0,0,0


Check categorical variables

In [18]:
categorical_columns = X_train_raw.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_columns = X_train_raw.select_dtypes(
    include=np.number
).columns.tolist()

print("Categorical columns:", len(categorical_columns))
print(categorical_columns)

print("\nNumerical columns:", len(numerical_columns))
print(numerical_columns)

Categorical columns: 13
['season', 'goal', 'shoton', 'shotoff', 'foulcommit', 'card', 'cross', 'corner', 'possession', 'league_name', 'country_name', 'home_team', 'away_team']

Numerical columns: 97
['stage', 'home_player_X1', 'home_player_X2', 'home_player_X3', 'home_player_X4', 'home_player_X5', 'home_player_X6', 'home_player_X7', 'home_player_X8', 'home_player_X9', 'home_player_X10', 'home_player_X11', 'away_player_X1', 'away_player_X2', 'away_player_X3', 'away_player_X4', 'away_player_X5', 'away_player_X6', 'away_player_X7', 'away_player_X8', 'away_player_X9', 'away_player_X10', 'away_player_X11', 'home_player_Y1', 'home_player_Y2', 'home_player_Y3', 'home_player_Y4', 'home_player_Y5', 'home_player_Y6', 'home_player_Y7', 'home_player_Y8', 'home_player_Y9', 'home_player_Y10', 'home_player_Y11', 'away_player_Y1', 'away_player_Y2', 'away_player_Y3', 'away_player_Y4', 'away_player_Y5', 'away_player_Y6', 'away_player_Y7', 'away_player_Y8', 'away_player_Y9', 'away_player_Y10', 'away_play

Check categorical consistency

In [19]:
for col in categorical_columns:
    print("=" * 80)
    print("COLUMN:", col)
    print("Training unique:", X_train_raw[col].nunique())
    print("Validation unique:", X_val_raw[col].nunique())
    print("Test unique:", X_test_raw[col].nunique())

    print("\nSample training values:")
    print(X_train_raw[col].dropna().unique()[:20])

COLUMN: season
Training unique: 6
Validation unique: 2
Test unique: 2

Sample training values:
['2008/2009' '2009/2010' '2010/2011' '2011/2012' '2012/2013' '2013/2014']
COLUMN: goal
Training unique: 8591
Validation unique: 2157
Test unique: 2479

Sample training values:
['<goal><value><comment>n</comment><stats><goals>1</goals><shoton>1</shoton></stats><event_incident_typefk>71</event_incident_typefk><elapsed>12</elapsed><player1>30872</player1><sortorder>0</sortorder><team>9823</team><id>375301</id><n>21</n><type>goal</type><goal_type>n</goal_type></value><value><comment>p</comment><stats><penalties>1</penalties></stats><event_incident_typefk>20</event_incident_typefk><elapsed>16</elapsed><player1>32118</player1><sortorder>0</sortorder><team>9823</team><id>375302</id><n>22</n><type>goal</type><goal_type>p</goal_type></value><value><comment>n</comment><stats><goals>1</goals><shoton>1</shoton></stats><event_incident_typefk>71</event_incident_typefk><elapsed>25</elapsed><player1>33101</p

Check numeric columns for impossible values

In [20]:
numeric_summary = X_train_raw[numerical_columns].describe().T

numeric_summary["Missing"] = X_train_raw[numerical_columns].isnull().sum()

display(numeric_summary)

,count,mean,std,min,25%,50%,75%,max,Missing
stage,18183.0,17.523181,10.253143,1.00,9.00,17.0,26.00,38.0,0
home_player_X1,16362.0,0.999389,0.027075,0.00,1.00,1.0,1.00,2.0,1821
home_player_X2,16362.0,2.052255,0.304834,0.00,2.00,2.0,2.00,8.0,1821
home_player_X3,16351.0,4.037368,0.306122,2.00,4.00,4.0,4.00,8.0,1832
home_player_X4,16351.0,6.059385,0.364422,2.00,6.00,6.0,6.00,8.0,1832
...,...,...,...,...,...,...,...,...,...
GBD,14162.0,3.648189,0.867440,1.45,3.20,3.3,3.75,11.0,4021
GBA,14162.0,4.353097,3.010189,1.12,2.50,3.4,5.00,34.0,4021
BSH,14161.0,2.497894,1.507793,1.04,1.67,2.1,2.62,17.0,4022
BSD,14161.0,3.660742,0.868272,1.33,3.25,3.4,3.75,13.0,4022


Date consistency

In [21]:
for df in [X_train_raw, X_val_raw, X_test_raw]:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("Missing dates:")
print("Train:", X_train_raw["date"].isna().sum())
print("Validation:", X_val_raw["date"].isna().sum())
print("Test:", X_test_raw["date"].isna().sum())

Missing dates:
Train: 0
Validation: 0
Test: 0


Missing-value handling

In [22]:
# Copy datasets
X_train = X_train_raw.copy()
X_val = X_val_raw.copy()
X_test = X_test_raw.copy()

# Define all 10 bookmaker odds columns to exclude from raw median imputation
BOOKMAKER_TRIPLETS = {
    "B365": ("B365H", "B365D", "B365A"),
    "BW":   ("BWH",   "BWD",   "BWA"),
    "IW":   ("IWH",   "IWD",   "IWA"),
    "LB":   ("LBH",   "LBD",   "LBA"),
    "PS":   ("PSH",   "PSD",   "PSA"),
    "WH":   ("WHH",   "WHD",   "WHA"),
    "SJ":   ("SJH",   "SJD",   "SJA"),
    "VC":   ("VCH",   "VCD",   "VCA"),
    "GB":   ("GBH",   "GBD",   "GBA"),
    "BS":   ("BSH",   "BSD",   "BSA"),
}
ODDS_COLS = [c for t in BOOKMAKER_TRIPLETS.values() for c in t]

# Numerical columns (EXCLUDING raw odds)
numeric_columns = [
    c for c in X_train.select_dtypes(include=np.number).columns.tolist()
    if c not in ODDS_COLS
]

# Categorical columns
categorical_columns = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical columns (excluding odds):", len(numeric_columns))
print("Categorical columns:", len(categorical_columns))

Numerical columns (excluding odds): 67
Categorical columns: 13


In [23]:
# Numerical imputation values learned from training data (odds remain NaN)
numeric_fill_values = X_train[numeric_columns].median()

# Categorical imputation values learned from training data
categorical_fill_values = {
    col: X_train[col].mode(dropna=True)[0]
    for col in categorical_columns
    if X_train[col].notna().any()
}

In [24]:
# Apply numerical imputation (only to non-odds numeric columns)
for col in numeric_columns:
    X_train[col] = X_train[col].fillna(numeric_fill_values[col])
    X_val[col] = X_val[col].fillna(numeric_fill_values[col])
    X_test[col] = X_test[col].fillna(numeric_fill_values[col])

# Apply categorical imputation
for col, value in categorical_fill_values.items():
    X_train[col] = X_train[col].fillna(value)
    X_val[col] = X_val[col].fillna(value)
    X_test[col] = X_test[col].fillna(value)

Verify missing values after preprocessing

In [28]:
print("Remaining missing values")

print("\nTraining:")
print(X_train.isnull().sum().sum())

print("Validation:")
print(X_val.isnull().sum().sum())

print("Test:")
print(X_test.isnull().sum().sum())

Remaining missing values

Training:
115851
Validation:
41700
Test:
45909


Verify the target wasn't accidentally changed

In [29]:
print("Training target:")
display(y_train.value_counts())

print("\nValidation target:")
display(y_val.value_counts())

print("\nTest target:")
display(y_test.value_counts())

Training target:


,count
target_outcome,
Home Win,8433
Away Win,5126
Draw,4624



Validation target:


,count
target_outcome,
Home Win,1753
Away Win,1147
Draw,973



Test target:


,count
target_outcome,
Home Win,1731
Away Win,1193
Draw,999


Final preprocessing summary

In [30]:
preprocessing_summary = pd.DataFrame({
    "Dataset": [
        "Training",
        "Validation",
        "Test"
    ],
    "Rows": [
        len(X_train),
        len(X_val),
        len(X_test)
    ],
    "Columns": [
        X_train.shape[1],
        X_val.shape[1],
        X_test.shape[1]
    ],
    "Missing Values": [
        X_train.isnull().sum().sum(),
        X_val.isnull().sum().sum(),
        X_test.isnull().sum().sum()
    ]
})

display(preprocessing_summary)

,Dataset,Rows,Columns,Missing Values
0,Training,18183,111,115851
1,Validation,3873,111,41700
2,Test,3923,111,45909


In [31]:
# Final ID alignment verification

assert len(train_ids) == len(X_train) == len(y_train)
assert len(val_ids) == len(X_val) == len(y_val)
assert len(test_ids) == len(X_test) == len(y_test)

# Verify the preprocessing stage did not change row order
assert train_ids.reset_index(drop=True).equals(
    train_df["id"].reset_index(drop=True)
)

assert val_ids.reset_index(drop=True).equals(
    val_df["id"].reset_index(drop=True)
)

assert test_ids.reset_index(drop=True).equals(
    test_df["id"].reset_index(drop=True)
)

print("ID and row alignment verified before saving.")

ID and row alignment verified before saving.


In [32]:
# Verify raw odds kept their genuine missing rates
for c in ["SJH", "GBH", "BSH"]:
    print(c, "NaN rate:", X_train[c].isna().mean().round(3))

SJH NaN rate: 0.132
GBH NaN rate: 0.221
BSH NaN rate: 0.221


Save preprocessed dataset

In [ ]:
PREPROCESSED_DIR = os.path.join(
    BASE_DIR,
    "preprocessed_dataset"
)

os.makedirs(PREPROCESSED_DIR, exist_ok=True)

X_train.to_csv(
    os.path.join(PREPROCESSED_DIR, "X_train_preprocessed.csv"), index=False
)

X_val.to_csv(
    os.path.join(PREPROCESSED_DIR, "X_validation_preprocessed.csv"), index=False
)

X_test.to_csv(
    os.path.join(PREPROCESSED_DIR, "X_test_preprocessed.csv"), index=False
)

y_train.to_csv(
    os.path.join(PREPROCESSED_DIR, "y_train.csv"), index=False
)

y_val.to_csv(
    os.path.join(PREPROCESSED_DIR, "y_validation.csv"), index=False
)

y_test.to_csv(
    os.path.join(PREPROCESSED_DIR, "y_test.csv"), index=False
)

# Save match IDs separately

train_ids.rename("id").to_csv(
    os.path.join(
        PREPROCESSED_DIR,
        "train_ids.csv"
    ),
    index=False
)

val_ids.rename("id").to_csv(
    os.path.join(
        PREPROCESSED_DIR,
        "validation_ids.csv"
    ),
    index=False
)

test_ids.rename("id").to_csv(
    os.path.join(
        PREPROCESSED_DIR,
        "test_ids.csv"
    ),
    index=False
)

print("Preprocessed datasets saved to:")
print(PREPROCESSED_DIR)

print("\nFiles:")
print(os.listdir(PREPROCESSED_DIR))

Preprocessed datasets saved to:
/content/drive/MyDrive/ML Assignment/preprocessed_dataset

Files:
['X_train_preprocessed.csv', 'X_validation_preprocessed.csv', 'X_test_preprocessed.csv', 'y_train.csv', 'y_validation.csv', 'y_test.csv', 'train_ids.csv', 'validation_ids.csv', 'test_ids.csv']
